# Employee Attrition Analytics — Data Preparation (ETL)

## 1. Introduction

This notebook covers the Extract, Transform, and Load (ETL) process using the IBM HR Analytics Employee Attrition dataset.

The goal is to audit employee data quality, prepare the attributes, and generate a clean, accurate dataset for exploratory data analysis (EDA) and Tableau dashboarding.

### Objectives

- Load and inspect the original dataset.
- Identify and resolve missing values, duplicates and inconsistent data.
- Identify redundant columns that do not analytical value.
- Engineer useful features tailored for analysing employee attrition.
- Validate the processed dataset.
- Export the prepared data for further analysis.

The source dataset will remain completely unaltered to guarantee full workflow reproducibility.

In [1]:
# Import Python libraries

import pandas as pd
from pathlib import Path


## 2. Data Extraction

The dataset was obtained from the IBM HR Analytics Employee Attrition dataset available on Kaggle.

The original CSV file is stored in the `data/raw` folder to preserve the source data.

In this stage, the dataset will be loaded into a pandas DataFrame. The first five rows will then be displayed to confirm that the file has loaded successfully and to provide an initial view of its structure.

**Source:** [IBM HR Analytics Employee Attrition Dataset](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)

In [2]:

# Identify the location of the original dataset
file_path = "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

# Load the CSV into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first five rows
df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 3. Initial Data Inspection

Before cleaning or transforming the dataset, it is important to understand its structure.

This section analyses:

- The amount of rows and columns.
- The titles of the available features.
- The data types utilised in each column.
- Whether any columns have missing values.

Doing these checks helps us find potential data quality issues and determine which transformations may be needed.

In [3]:

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Show all column titles
print("\nColumn names:")
print(df.columns.tolist())

# Inspect data types and non-null values
print("\nDataset information:")
df.info()


Dataset shape: (1470, 35)

Column names:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   ob

### 3.1 Findings and Interpretation

The first inspection identified:
**1,470 employee records and 35 columns**

The dataset consists of:

- **26 integer columns (`int64`)**, including Age, MonthlyIncome, YearsAtCompany and JobSatisfaction.
- **9 object columns (`object`)**, including Attrition, Department, JobRole and OverTime.
- **No missing values identified during the first inspection**, as every column contains 1,470 non-null values.

The dataset consists of several features such as overtime, job satisfaction, work-life balance and length of employment. These are relevant in order to understand employee attrition. 

Although no missing values were identified, The work is not finished as duplicate records need to be checked for, as well as constant columns, categorical consistency and potential data quality problems.

**Business relevance:** Reviewinf the dataset's structure helps determine which employee characteristics can be investigated and whether the data can support dependable HR insights.

**Ethical consideration:** Employee-related data must be handled responsibly. Although this project uses a publicly available synthetic dataset, its findings should not be utilised to make assumptions about individual employees or establish causal relationships.

## 4. Data Quality Assessment

Data quality inspections are necessary to make sure that the dataset is suitable for analysis.

This section examines:

- Missing values that could influence calculations.
- Duplicate rows that could inflate employee counts.
- Duplicate employee identifiers that could indicate repeated entries.

Finding these issues before transformation helps ensure the analysis that follows, is both accurate and reliable.

In [4]:
# Check for missing values in each individual column
missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

# Look for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Identify duplicate employee identifiers
print("Duplicate employee IDs:", df["EmployeeNumber"].duplicated().sum())


Total missing values: 0
Duplicate rows: 0
Duplicate employee IDs: 0


### 4.1 Data Quality Findings and Interpretation

The data quality assessment identified the following results:

- **Missing values:** 0
- **Duplicate records:** 0
- **Duplicate employee identifiers:** 0

The results show that the dataset has no missing values or duplicate employee records in the checks performed.

Since the checks found no missing or duplicate records, neither imputation or duplicate removal was needed. Preserving the original records avoids unnecessary alterations to the data that could have an affect on the accuracy of the analysis.

However, further validation is required to identify constant columns, inspect categorical values and check whether numerical features contain invalid or inconsistent values.

**Business relevance:** These checks help ensure missing values or duplicate records will not distort employee counts and attrition calculations.

## 5. Finding Constant Columns

Constant columns contain only one unique value across all records.

Since they do not vary between employees, they generally provide limited value when analysing factors related to employee attrition.

This step identifies such columns so that any decisions to remove them from the processed dataset can be justified and documented.

In [5]:

# Count the number of unique values in each column
unique_counts = df.nunique()

# Identify columns containing only one unique value
constant_columns = unique_counts[unique_counts == 1]

print("Constant columns:")
print(constant_columns)

# Display the actual value stored in each constant column
for column in constant_columns.index:
    print(f"{column}: {df[column].iloc[0]}")


Constant columns:
EmployeeCount    1
Over18           1
StandardHours    1
dtype: int64
EmployeeCount: 1
Over18: Y
StandardHours: 80


### 5.1 Findings and Interpretation

The inspection identified three constant columns:

- **EmployeeCount:** Contains the value "1" for every employee.
- **Over18:** Contains the value "Y" for every employee.
- **StandardHours:** Contains the value "80" for every employee.

These columns contain no variation across the 1,470 employee records and therefore offer little analytical value when investigating employee attrition.

The decision was made to remove these columns from the processed dataset, but remain unchanged in the original CSV file.

Removing these features will make the dataset simpler without losing required information for the project's business questions.

**Business relevance:** Taking away non-informative features such as these constant columns allow the analysis to focus on employee characteristics that vary, such as overtime, job satisfaction and years at the company.

In [6]:
# Create a working copy to preserve the original DataFrame
df_clean = df.copy()

# Remove columns that contain the same value for every employee
df_clean = df_clean.drop(columns=constant_columns.index)

# Check the updated dataset dimensions
print("Original dataset:", df.shape)
print("Processed dataset:", df_clean.shape)

# Confirm the columns were removed
print("\nRemoved columns:", constant_columns.index.tolist())


Original dataset: (1470, 35)
Processed dataset: (1470, 32)

Removed columns: ['EmployeeCount', 'Over18', 'StandardHours']


## 6. Categorical Data Validation

Categorical variables consists of labels or groups rather than continuous numerical measurements.

This section reviews the distinct values in key categorical columns to spot possible inconsistencies, including unexpected labels, spelling differences or extra spaces.

Validating these values is important because inconsistent categories could skew employee counts, attrition rates and could affect Tableau visualisations.

The investigation will show whether any standardisation is necessary before further transformations.

In [7]:
# Identify categorical columns in the processed dataset
categorical_columns = df_clean.select_dtypes(include="object").columns

# Display distinct values for each categorical column
for column in categorical_columns:
    print(f"\n{column}:")
    print(df_clean[column].value_counts(dropna=False))



Attrition:
Attrition
No     1233
Yes     237
Name: count, dtype: int64

BusinessTravel:
BusinessTravel
Travel_Rarely        1043
Travel_Frequently     277
Non-Travel            150
Name: count, dtype: int64

Department:
Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64

EducationField:
EducationField
Life Sciences       606
Medical             464
Marketing           159
Technical Degree    132
Other                82
Human Resources      27
Name: count, dtype: int64

Gender:
Gender
Male      882
Female    588
Name: count, dtype: int64

JobRole:
JobRole
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               52
Name: count, dtype: int64

MaritalStatus:
MaritalStatus
Married  

### 6.1 Findings and Interpretation

The categorical data review identified eight categorical variables in the processed dataset.

The categories appear consistently labelled, with no obvious spelling differences or unexpected values that need correcting.

Key observations:

- **Attrition:** 237 employees (16.1%) left the organisation, while 1,233 (83.9%) remained.
- **Department:** Research & Development is the largest department, with 961 employees.
- **JobRole:** Sales Executive is the most common role, with 326 employees.
- **OverTime:** 416 employees work overtime, while 1,054 do not.
- **BusinessTravel:** Most employees (1,043) travel rarely.

As no category inconsistencies were found, no category standardisation is needed at this stage.

**Business relevance:** These variables will support comparisons of attrition rates across departments, job roles and working conditions.

**Analytical limitation:** Employee counts alone do not reveal which groups have the highest attrition rates. Further analysis should compare the proportion of employees who leave in each group.

**Ethical consideration:** Demographic variables, including gender and marital status should be interpreted responsibly to avoid assumptions and recommendations that are discriminatory.

In [8]:
# Validate the expected categories
expected_values = {
    "Attrition": {"Yes", "No"},
    "OverTime": {"Yes", "No"}
}

for column, expected in expected_values.items():
    actual = set(df_clean[column].unique())
    print(f"{column} valid categories: {actual == expected}")


Attrition valid categories: True
OverTime valid categories: True


## 7. Numerical Data Validation

Before analysis numerical data must be checked for invalid values and inconsistencies.

This part looks at key numerical variables, such as employee age, monthly income, job satisfaction, work-life balance and length of employment.

The objective is to flag values that might be invalid, such as negative income, unrealistic ages or employment durations that go beyond an employee's total work experience.

These checks ensure the suitability of the dataset for a reliable analysis and visualisation.

In [9]:
# Select key numerical columns for inspection
numerical_columns = [
    "Age",
    "MonthlyIncome",
    "JobSatisfaction",
    "WorkLifeBalance",
    "TotalWorkingYears",
    "YearsAtCompany"
]

# Display summary statistics
df_clean[numerical_columns].describe()


,Age,MonthlyIncome,JobSatisfaction,WorkLifeBalance,TotalWorkingYears,YearsAtCompany
count,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000
mean,36.923810,6502.931293,2.728571,2.761224,11.279592,7.008163
std,9.135373,4707.956783,1.102846,0.706476,7.780782,6.126525
min,18.000000,1009.000000,1.000000,1.000000,0.000000,0.000000
25%,30.000000,2911.000000,2.000000,2.000000,6.000000,3.000000
50%,36.000000,4919.000000,3.000000,3.000000,10.000000,5.000000
75%,43.000000,8379.000000,4.000000,3.000000,15.000000,9.000000
max,60.000000,19999.000000,4.000000,4.000000,40.000000,40.000000


### 7.1 Numerical Data Findings

The descriptive statistics provide an overview of the numerical variables relevant to employee attrition.

Key observations include:

- **Age:** Employees range from 18 to 60 years old, with an average age of approximately 36.9 years.
- **Monthly Income:** Values range from 1,009 to 19,999, with an average of approximately 6,502.9. The dataset does not establish a currency, so these values are reported without a currency symbol.
- **Job Satisfaction:** Scores range from 1 to 4, with an average of 2.73.
- **Work-Life Balance:** Scores range from 1 to 4, with an average of 2.76.
- **Total Working Years:** Employees have between 0 and 40 years of working experience.
- **Years at Company:** Employees have worked at the organisation for between 0 and 40 years.

The value ranges seem reasonable, and the summary statistics did not reveal any obvious invalid values.

However, further research is needed to identify negative values, scores outside their expected ranges and any mismatch between years at the company and total work experience.

**Business relevance:** These variables can help explore how employee characteristics, satisfaction levels and employment tenure, are associated with attrition.

**Analytical limitation:** Descriptive statistics provide an overview of the workforce but do not establish relationships between these variables or prove causation.

In [10]:
# Check for invalid numerical values
validation_checks = {
    "Invalid ages": (~df_clean["Age"].between(18, 100)).sum(),
    "Non-positive monthly income": (df_clean["MonthlyIncome"] <= 0).sum(),
    "Invalid job satisfaction scores": (
        ~df_clean["JobSatisfaction"].between(1, 4)
    ).sum(),
    "Invalid work-life balance scores": (
        ~df_clean["WorkLifeBalance"].between(1, 4)
    ).sum(),
    "Negative total working years": (
        df_clean["TotalWorkingYears"] < 0
    ).sum(),
    "Negative years at company": (
        df_clean["YearsAtCompany"] < 0
    ).sum(),
    "Company tenure exceeds total experience": (
        df_clean["YearsAtCompany"] >
        df_clean["TotalWorkingYears"]
    ).sum()
}

for check, count in validation_checks.items():
    print(f"{check}: {count}")


Invalid ages: 0
Non-positive monthly income: 0
Invalid job satisfaction scores: 0
Invalid work-life balance scores: 0
Negative total working years: 0
Negative years at company: 0
Company tenure exceeds total experience: 0


### 7.3 Numerical Validation Results

Seven checks were carried out to evaluate the consistency and reliability of key numerical variables. 

**No invalid records were found in any of the checks.**

The checks confirmed that:

- Employee ages are within the defined validation range of 18–100.
- All monthly income values are positive.
- Job satisfaction and work-life balance scores are within the expected 1–4 scale.
- Total working years and years at the company contain no negative values.
- Each employee's years at the company are no greater than their total working experience.

**Data preparation decision:** Since the validation checks found no violations, all numerical values retained without any modifications.

**Business relevance:** These findings strengthen confidence in the reliability of the numerical variables used for workforce reporting and attrition analysis.

**Limitation:** Passing these checks does not ensure that every value is correct or that the dataset is free from all possible inconsistencies.

## 8. Feature Engineering

Feature engineering creates new variables from existing data to make analysis and reporting more meaningful and informative.

Following the completion of the data quality and validation checks, three new features will be addedd:

- **Attrition_Flag:** Turns the Attrition column into a binary variable, where 1 represents an employee who left and 0 for an employee who stayed.
- **Age_Group:** Places employees into age bands to support comparisons between different age groups.
- **Tenure_Group:** Categorises employees according to their years of work at the organisation.

These transformations will simplify attrition-rate calculations and compare employee groups utilising Python and Tableau.

The original Age, YearsAtCompany and Attrition columns will remain unchanged to retain the underlying information.

**Ethical consideration:** Group-level patterns should not be viewed as evidence that an individual's age or work history causes attrition. Age-related insights must never be used to support discriminatory employment decisions.

In [11]:
# Create a binary attrition indicator
df_clean["Attrition_Flag"] = df_clean["Attrition"].map({
    "Yes": 1,
    "No": 0
})

# Assign employees to age groups
df_clean["Age_Group"] = pd.cut(
    df_clean["Age"],
    bins=[17, 25, 35, 45, 55, float("inf")],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)

# Create employee tenure groups
df_clean["Tenure_Group"] = pd.cut(
    df_clean["YearsAtCompany"],
    bins=[-1, 2, 5, 10, 20, float("inf")],
    labels=["0-2 years", "3-5 years", "6-10 years",
            "11-20 years", "21+ years"]
)

# Preview the new columns
df_clean[
    ["Attrition", "Attrition_Flag", "Age",
     "Age_Group", "YearsAtCompany", "Tenure_Group"]
].head(10)


,Attrition,Attrition_Flag,Age,Age_Group,YearsAtCompany,Tenure_Group
0,Yes,1,41,36-45,6,6-10 years
1,No,0,49,46-55,10,6-10 years
2,Yes,1,37,36-45,0,0-2 years
3,No,0,33,26-35,8,6-10 years
4,No,0,27,26-35,2,0-2 years
5,No,0,32,26-35,7,6-10 years
6,No,0,59,56+,1,0-2 years
7,No,0,30,26-35,1,0-2 years
8,No,0,38,36-45,9,6-10 years
9,No,0,36,36-45,7,6-10 years


### 8.1 Feature Engineering Findings

Three additional features were created successfully to help with the employee attrition analysis.

**Attrition_Flag** converts employee attrition status into numerical values (1 for employees who left and 0 for employees who stayed). This makes it easier to calculate attrition counts and rates in a straightforward manner.

**Age_Group** sort employees into five age groups: 18–25, 26–35, 36–45, 46–55 and 56+.

**Tenure_Group** divides employees into five bands based on their years at the company: 0–2, 3–5, 6–10, 11–20 and 21+ years.

The initial preview shows that the new features were  assigned to the displayed records as intended.

These transformations retain the original columns,so that the data can be analysed both as individual values and as groups.

**Business relevance:** The new features will make it easier to compare attrition across employee groups in Python and Tableau.

**Analytical and ethical limitations:** The age and tenure groups were defined for this research ans were not classifications provided by the original dataset. Group-level associations must not be interpreted as causal relationships or utilised to support discriminatory employment decisions.

In [12]:
# Confirm the processed dataset dimensions
print("Processed dataset shape:", df_clean.shape)

# Check for missing values in the newly created features
new_features = ["Attrition_Flag", "Age_Group", "Tenure_Group"]

print("\nMissing values in new features:")
print(df_clean[new_features].isnull().sum())

# Verify that the attrition flag matches the original labels
expected_flag = df_clean["Attrition"].map({"Yes": 1, "No": 0})

print("\nAttrition flag matches original:",
      df_clean["Attrition_Flag"].equals(expected_flag))

# Count the number of employees who left the organisation
print("Employees who left:", df_clean["Attrition_Flag"].sum())

# Show the distribution of the new groups
print("\nAge group distribution:")
print(df_clean["Age_Group"].value_counts().sort_index())

print("\nTenure group distribution:")
print(df_clean["Tenure_Group"].value_counts().sort_index())


Processed dataset shape: (1470, 35)

Missing values in new features:
Attrition_Flag    0
Age_Group         0
Tenure_Group      0
dtype: int64

Attrition flag matches original: True
Employees who left: 237

Age group distribution:
Age_Group
18-25    123
26-35    606
36-45    468
46-55    226
56+       47
Name: count, dtype: int64

Tenure group distribution:
Tenure_Group
0-2 years      342
3-5 years      434
6-10 years     448
11-20 years    180
21+ years       66
Name: count, dtype: int64


### 8.3 Feature Engineering Validation Results

The newly created features were checked to confirm that the transformations were applied correctly.

The processed dataset contains **1,470 employee records across 35 columns**, including the three added features.

The checks confirmed that:

- Attrition_Flag, Age_Group or Tenure_Group contain no missing values.
- Attrition_Flag correctly matches the original Attrition labels.
- The number of workers who left is still 237.
- All 1,470 employees were assigned to an age group and a tenure group.

The largest age band is **26–35 years**, consisting of 606 employees. The largest tenure group is **6–10 years**, with 448 employees.

These results show that the new features are populated consistently and that all employee records were retained during the transformation.

**Business relevance:** The processed dataset is ready to support group-level attrition comparisons in Python and Tableau.

**Limitation:** The group distributions describe workforce composition, not the likelihood of attrition in each group. Therefore, attrition rates need to be calculated separately for each group before coming to a conclusion.

## 9. Final Dataset Checks

Following data cleaning and feature engineering, a final validation is performed to confirm that the processed dataset is complete, accurate, and ready for further analysis.

This stage checks that:

- Every original employee record has been retained.
- Employee identifiers remain unique.
- No missing values have been introduced.
- The three constant columns have been removed.
- All three engineered features are included.
- The number of employees who left and the overall attrition rate match the original dataset.

These checks help confirm that the processed data is ready for explaratory data analysis (EDA) and Tableau dashboards.

In [13]:
# Make a comparison of original and processed employee counts
print("Original employee count:", len(df))
print("Processed employee count:", len(df_clean))

# Check for missing values and duplicate employee Identifiers
print("\nMissing values:", df_clean.isnull().sum().sum())
print("Duplicate employee IDs:",
      df_clean["EmployeeNumber"].duplicated().sum())

# Ensure constant columns were removed
removed_columns = ["EmployeeCount", "Over18", "StandardHours"]

print("\nConstant columns successfully removed:",
      all(col not in df_clean.columns for col in removed_columns))

# Verify that engineered features are included
new_columns = ["Attrition_Flag", "Age_Group", "Tenure_Group"]

print("Engineered features present:",
      all(col in df_clean.columns for col in new_columns))

# Compare original and processed attrition counts
original_attrition = (df["Attrition"] == "Yes").sum()
processed_attrition = df_clean["Attrition_Flag"].sum()

print("\nOriginal attrition count:", original_attrition)
print("Processed attrition count:", processed_attrition)

# Calculate overall attrition rate
attrition_rate = df_clean["Attrition_Flag"].mean() * 100

print(f"Overall attrition rate: {attrition_rate:.2f}%")


Original employee count: 1470
Processed employee count: 1470

Missing values: 0
Duplicate employee IDs: 0

Constant columns successfully removed: True
Engineered features present: True

Original attrition count: 237
Processed attrition count: 237
Overall attrition rate: 16.12%


### 9.1 Final Validation Findings and Interpretation

The final validation confirmed that the processed dataset has remained intact following data cleaning and feature engineering.

**Validation results:**

- **Employee records:** All 1,470 records from the original dataset were preserved.
- **Missing values:** No missing values were found.
- **Duplicate employee identifiers:** No duplicates were identified.
- **Constant columns:** EmployeeCount, Over18 and StandardHours were removed.
- **Engineered features:** Attrition_Flag, Age_Group and Tenure_Group are included.
- **Attrition count:** Both the original and processed datasets show that there are 237 employees who left.
- **Overall attrition rate:** 16.12%.

The unchanged employee counts and attrition totals show that the transformations preserved these key workforce metrics.

**Business relevance:** The validated dataset provides a consistent foundation for calculating workforce KPIs, exploring attrition patterns and creating interactive Tableau dashboards.

**Data quality conclusion:** The checks found no missing values, duplicate employee ID's, or breaches of the selected numerical validation rules. That means that the dataset is suitable for the planned exploratory analysis, though these checks cannot make the confirmation that every value is accurate.

## 10. Exporting the Prepared Dataset

Following successful validation, the processed dataset will be exported as a CSV file ready to be used in the exploratory data analysis as well as Tableau.

The file will be placed in a versioned directory (`data/processed/v1/`) to support reproducibility and simplify future updates.

The original source CSV will remain untouched in `data/raw/`.

The exported file will contain 1,470 employee records, 32 retained original columns and three newly engineered features.

In [14]:
# Create the versioned output directory
output_folder = Path("../data/processed/v1")
output_folder.mkdir(parents=True, exist_ok=True)

# Define the output file path
output_path = output_folder / "employee_attrition_cleaned.csv"

# Export the processed dataset
df_clean.to_csv(output_path, index=False)

# Verify the export
print("Exported file:", output_path)
print("Exported shape:", df_clean.shape)
print("File exists:", output_path.exists())


Exported file: ..\data\processed\v1\employee_attrition_cleaned.csv
Exported shape: (1470, 35)
File exists: True


In [15]:
# Reload the exported CSV
df_exported = pd.read_csv(output_path)

# Make a comparison between the exported file and the prepared dataset
print("Exported dataset shape:", df_exported.shape)
print("Employee counts match:", len(df_exported) == len(df_clean))
print("Column names match:",
      df_exported.columns.tolist() == df_clean.columns.tolist())

print("Attrition counts match:",
      df_exported["Attrition_Flag"].sum() ==
      df_clean["Attrition_Flag"].sum())

print("Missing values:", df_exported.isnull().sum().sum())


Exported dataset shape: (1470, 35)
Employee counts match: True
Column names match: True
Attrition counts match: True
Missing values: 0


## 11. ETL Conclusion

The Extract, Transform and Load (ETL) process has successfully prepared the IBM HR Analytics Employee Attrition dataset for exploratory data analysis (EDA) and Tableau visualisation.

### Summary of Data Preparation

The source dataset contained **1,470 employee records and 35 columns**.

Data quality checks found:

- No missing values.
- No duplicate records or employee identifiers.
- No invalid values under the selected numerical validation checks.
- Three constant columns: EmployeeCount, Over18 and StandardHours.

These three constant columns were removed from the processed dataset because they did not provide any variation and were of no value for the planned analysis.

Three new features were created:

- **Attrition_Flag:** A binary indicator supporting attrition rate calculations.
- **Age_Group:** Five bands for comparing employee by age.
- **Tenure_Group:** Five categories for comparing employees by length of service.

The final processed dataset contains **1,470 employee records and 35 columns**.

### Validation and Export

Final checks confirmed that all employee records were retained, no missing values were introduced and the number of employees who left stayed **237**, representing an overall attrition rate of **16.12%**.

The processed dataset was saved to:

`data/processed/v1/employee_attrition_cleaned.csv`

The original dataset remains unchanged in `data/raw/`.

### Ethical and Analytical Limitations

The project uses a publicly available synthetic HR dataset. Although the records are fictional, responsible HR analytics principles still apply.

Automated ETL checks cannot rule out all systematic errors in the source data. Furthermore, relationships observed in the dataset do not establish that they cause employee attrition.

The engineered age and tenure groups are analytical aggregates and should not be used to inform real-world, discriminatory employment decisions.

### Next Stage

The validated dataset will now advance to the Exploratory Data Analysis (EDA) notebook to investigate the association between attrition patterns and variables like departments, job roles, overtime hours, job satisfaction and employment tenure.

Statistical hypothesis testing will be utilized to evaluate selected associations, while Tableau dashboards will map out the final findings for non-technical stakeholders.